# Year shifts and validation design (no model fitting)

Investigate the claim that 2022 is unusually difficult and whether leave-one-year-out
(LOYO) validation provides a more useful estimate. This notebook computes raw-input
and label EDA, audits chronological/LOYO partitions, and optionally reads **existing**
TCN scores. It trains no model, loads no weights and changes no frozen recipe.

The task is a scheduling surrogate with supplied planning-week exogenous inputs.
Nevertheless, the official prediction cases start after the labeled period.
LOYO tests transfer to an omitted year using other years, potentially including
later regimes; forward validation tests transfer to a later year from earlier data.
They answer different questions. A higher validation score is not by itself better evidence.

Historical 2020/2021 years and 2022 already have project exposure. Test inputs are
used only for descriptive EDA here; there are no test labels. None of these
summaries changes model preprocessing or hyperparameters.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import re
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
import yaml
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data/kernel/Unit_commitment_decisions.csv").exists())
STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
OUT = ROOT / "outputs/year_diagnostics" / STAMP
OUT.mkdir(parents=True, exist_ok=False)
HOURS = 168
FILES = {
    "labels": ROOT / "data/kernel/Unit_commitment_decisions.csv",
    "mapping": ROOT / "prediction_mapping.csv",
    "template": ROOT / "sample_submission.csv",
    "prices": ROOT / "data/kernel/Historical_day_ahead_price_2015_2025.csv",
    "inflows": ROOT / "data/kernel/Historical_inflow_1958_2025.csv",
    "volumes": ROOT / "data/kernel/Historical_volume_2015_2024.csv",
    "water_values": ROOT / "data/kernel/Synthetic_water_value_2015_2024.csv",
    "topology": ROOT / "data/extended/Tokke_Vinje_topology.yaml",
}
train = pd.read_csv(FILES["labels"])
test = pd.read_csv(FILES["mapping"])
for frame in (train, test):
    frame["starttime"] = pd.to_datetime(frame["starttime"], utc=True, errors="raise")
    assert frame["Run No"].is_unique and frame.starttime.is_unique
    assert (frame.starttime == frame.starttime.dt.normalize()).all()
assert len(train) == 2922 and len(test) == 725
LABELS = [c for c in train if re.fullmatch(r"result_committed_.+_t\d+", c)]
assert len(LABELS) == 2352
assert pd.read_csv(FILES["template"], nrows=0).columns.tolist() == ["Run No"] + LABELS
assert np.isin(train[LABELS].to_numpy(), [0, 1]).all()
GENERATORS = list(dict.fromkeys(re.fullmatch(r"result_committed_(.+)_t(\d+)", c)[1] for c in LABELS))
assert len(GENERATORS) == 14
cases = pd.concat([train[["Run No", "starttime"]], test[["Run No", "starttime"]]], ignore_index=True)
assert cases["Run No"].is_unique and cases.starttime.is_unique
assert cases.starttime.diff().dropna().eq(pd.Timedelta(days=1)).all()
FIRST, END = cases.starttime.min(), cases.starttime.max() + pd.Timedelta(hours=HOURS)
YEARS = sorted(cases.starttime.dt.year.unique())


def read_series(key):
    frame = pd.read_csv(FILES[key])
    frame.index = pd.DatetimeIndex(pd.to_datetime(frame.pop("date"), utc=True, errors="raise"))
    assert frame.index.is_unique
    frame = frame.sort_index().loc[FIRST:END].apply(pd.to_numeric, errors="raise")
    assert np.isfinite(frame.to_numpy()).all()
    return frame


tables = {k: read_series(k) for k in ("prices", "inflows", "volumes", "water_values")}
expected = pd.date_range(FIRST, END, freq="h", inclusive="left")
for key in ("prices", "inflows"):
    assert tables[key].loc[tables[key].index < END].index.equals(expected)
with FILES["topology"].open(encoding="utf-8") as stream:
    topology = yaml.safe_load(stream)
RESERVOIRS = tables["volumes"].columns.tolist()
assert len(RESERVOIRS) == 17
capacity = np.array([topology["model"]["reservoir"][r]["max_vol"] for r in RESERVOIRS])
assert (capacity > 0).all()
print("Output directory:", OUT)


## 1. Compare the inputs available to each scheduling case

Use storage at S, terminal values at S+168h and price/inflow during [S,S+168h).
These are simple descriptive summaries, **not the full 913/182 model feature set**.
Reservoir inflows exclude the additional river series to avoid presenting it as
another independent reservoir supply. Mean local water value is an unweighted
reservoir summary, not a globally converted economic water cost.

Annual boxes describe case distributions (10th-90th percentile whiskers), not
independent hourly samples. Daily overlapping horizons induce dependence.
2024 has only the supplied starts through December 25.

In [ ]:
starts = pd.DatetimeIndex(cases.starttime)
# Timestamp lookup rather than file-row offsets; later 2025 quarter-hours are excluded.
times = pd.DatetimeIndex(np.repeat(starts.to_numpy(), HOURS)) + pd.to_timedelta(
    np.tile(np.arange(HOURS), len(cases)), unit="h")
price = tables["prices"].reindex(times).to_numpy().reshape(len(cases), HOURS)
inflow = tables["inflows"][RESERVOIRS].sum(axis=1).reindex(times).to_numpy().reshape(len(cases), HOURS)
initial = tables["volumes"][RESERVOIRS].reindex(starts).to_numpy()
terminal = tables["water_values"][RESERVOIRS].reindex(starts + pd.Timedelta(hours=HOURS)).to_numpy()
for values in (price, inflow, initial, terminal):
    assert np.isfinite(values).all()
features = cases.copy()
features["year"] = starts.year
features["month"] = starts.month
features["weekly_mean_price_EUR_MWh"] = price.mean(axis=1)
features["weekly_p95_price_EUR_MWh"] = np.quantile(price, .95, axis=1)
features["negative_price_fraction"] = (price < 0).mean(axis=1)
features["mean_terminal_local_value_EUR_MWh"] = terminal.mean(axis=1)
features["initial_capacity_weighted_fill"] = initial.sum(axis=1) / capacity.sum()
features["natural_reservoir_inflow_week_Mm3"] = inflow.sum(axis=1) * 3600 / 1e6
features["price_to_mean_terminal_value"] = price.mean(axis=1) / np.maximum(terminal.mean(axis=1), 1e-6)
COLUMNS = [c for c in features if c not in {"Run No", "starttime", "year", "month"}]
year_summary = features.groupby("year")[COLUMNS].agg(["mean", "median", "min", "max"])
features.to_csv(OUT / "case_input_summaries.csv", index=False)
year_summary.to_csv(OUT / "year_input_summary.csv")
display(year_summary.round(3))

colors = ["#d59022" if y == 2022 else "#258a89" if y >= 2023 else "#8fa7be" for y in YEARS]
plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
fig, axes = plt.subplots(2, 2, figsize=(13, 8), layout="constrained")
for ax, column, title, unit in zip(axes.flat,
        ["weekly_mean_price_EUR_MWh", "mean_terminal_local_value_EUR_MWh",
         "initial_capacity_weighted_fill", "natural_reservoir_inflow_week_Mm3"],
        ["Planning-week electricity price", "Terminal local water values (17-reservoir mean)",
         "Initial total storage / total capacity", "Planning-week natural reservoir inflow"],
        ["EUR/MWh", "EUR/MWh", "Fraction (capacity weighted)", "Million m3 per case week"]):
    boxes = ax.boxplot([features.loc[features.year == y, column] for y in YEARS],
                      tick_labels=YEARS, whis=(10, 90), showfliers=False, patch_artist=True,
                      medianprops={"color": "#172b42", "linewidth": 1.5})
    for box, color in zip(boxes["boxes"], colors):
        box.set_facecolor(color)
    ax.set(title=title, ylabel=unit)
    ax.grid(axis="y", alpha=.18)
fig.suptitle("Input regimes by scheduling-case start year", fontsize=17, fontweight="bold")
fig.legend(handles=[Patch(color="#8fa7be", label="Labeled 2015-2021"),
                    Patch(color="#d59022", label="Labeled 2022"),
                    Patch(color="#258a89", label="Unlabeled test inputs")],
           loc="outside lower center", ncol=3, frameon=False)
fig.savefig(OUT / "year_input_regimes.png", dpi=160)
plt.show()


## 2. Separate target changes from measured model difficulty

ON fractions are EDA of supplied training labels, never test labels or predictor
inputs. A changed class balance alone does not establish lower ROC-AUC.

If the original experiment evidence is available, the score panel reads saved
v22 and v32 chronological results. It does **not** fit or evaluate a new model.
Those recipes have 2022 diagnostics; the canonical v35 6-hour recipe does not.
The 2020/2021 scores cover 360/359 cases, versus 365 in the old 2022 diagnostic;
epoch budgets and fitting histories also differ. This is context, not controlled
identification of an intrinsically harder year.

In [ ]:
on_rates = pd.DataFrame(index=range(2015, 2023), columns=GENERATORS, dtype=float)
for year in on_rates.index:
    rows = train.loc[train.starttime.dt.year == year]
    for generator in GENERATORS:
        columns = [f"result_committed_{generator}_t{h}" for h in range(HOURS)]
        on_rates.loc[year, generator] = rows[columns].to_numpy().mean()
on_rates.to_csv(OUT / "generator_on_fraction_by_year.csv", index_label="year")

historical = []
reference = ROOT / "outputs/tcn_development/20261027T001630Z_resolution/final_summary.json"
comparison = ROOT / "outputs/tcn_development/20261027T001630Z_resolution/development_comparison.csv"
if reference.exists() and comparison.exists():
    saved = json.loads(reference.read_text())
    development = pd.read_csv(comparison).set_index("name")
    print("Available development rows:", development.index.tolist())
    for recipe, holdout_key in [("local_terminal_bestpoint16", "selected_2022_auc"),
                                 ("local_hydraulic16", "baseline_2022_auc")]:
        matched = [name for name in development.index if name == recipe or name == "baseline_" + recipe]
        if not matched:
            # Some reports use baseline identity in the display name.
            matched = [name for name in development.index if recipe in name]
        if matched:
            row = development.loc[matched[0]]
            for year in (2020, 2021):
                historical.append({"recipe": recipe, "year": year, "micro_auc": float(row[f"auc{year}"]),
                                   "scoring_cases": 360 if year == 2020 else 359, "source": str(comparison.relative_to(ROOT))})
        historical.append({"recipe": recipe, "year": 2022, "micro_auc": saved[holdout_key],
                           "scoring_cases": 365, "source": str(reference.relative_to(ROOT))})
historical = pd.DataFrame(historical)
historical.to_csv(OUT / "saved_chronological_scores.csv", index=False)
fig, axes = plt.subplots(1, 2, figsize=(15, 7), layout="constrained", gridspec_kw={"width_ratios": [1.35, 1]})
im = axes[0].imshow(on_rates.T, vmin=0, vmax=1, aspect="auto", cmap="viridis")
axes[0].set(xticks=range(8), xticklabels=on_rates.index, yticks=range(14), yticklabels=GENERATORS,
            title="Observed ON fraction by unit and case year\n(training labels only; not an error metric)")
fig.colorbar(im, ax=axes[0], label="ON fraction", shrink=.75)
if not historical.empty:
    for (recipe, rows), color in zip(historical.groupby("recipe"), ["#8fa7be", "#258a89"]):
        axes[1].plot(rows.year, rows.micro_auc, "o-", label=recipe, color=color)
        for row in rows.itertuples():
            axes[1].annotate(f"{row.micro_auc:.5f}", (row.year, row.micro_auc), xytext=(0, 8),
                             textcoords="offset points", ha="center", fontsize=9)
    axes[1].legend(loc="lower right", fontsize=8)
    axes[1].set(xticks=[2020, 2021, 2022], ylim=(historical.micro_auc.min()-.002, historical.micro_auc.max()+.002))
else:
    axes[1].text(.5, .5, "Historical output files unavailable.\nNo scores inferred.", ha="center", transform=axes[1].transAxes)
axes[1].set(title="Existing chronological TCN results\n2022 is a previously exposed diagnostic", ylabel="Micro ROC-AUC")
axes[1].grid(alpha=.18)
fig.suptitle("Different years do not imply a model-independent difficulty ranking", fontsize=16, fontweight="bold")
fig.savefig(OUT / "year_targets_and_saved_scores.png", dpi=160)
plt.show()


## 3. Compare forward and LOYO partitions, with horizon purging

For a like-for-like prospective comparison, both strategies below score cases
whose complete 168-hour horizons fit inside the held year. This reproduces the
current development case counts for 2020/2021; it gives 359 cases for 2022 rather
than the older 365-case diagnostic. Do not silently substitute one score for the other.

Forward fitting requires training_end <= validation_start. LOYO fitting requires
training_end <= validation_start **or** training_start >= validation_end. A simple
year group alone leaves overlapping previous-year horizons in fitting data.
If validation instead includes the last six starts of December, the right-hand
purge must also extend into January. Fit all learned preprocessing only on fitting
cases, and use separate inner data for stopping/tuning. These are partitions only;
no model is fitted or scored below.

In [ ]:
def year_partition(year, strategy):
    start = pd.Timestamp(f"{year}-01-01", tz="UTC")
    end = pd.Timestamp(f"{year+1}-01-01", tz="UTC")
    starts = train.starttime
    ends = starts + pd.Timedelta(hours=HOURS)
    validation = (starts >= start) & (ends <= end)
    assert validation.any()
    left, right = starts[validation].min(), ends[validation].max()
    past = ends <= left
    future = starts >= right
    fitting = past if strategy == "forward" else past | future
    assert not (fitting & validation).any()
    assert ((ends[fitting] <= left) | (starts[fitting] >= right)).all()
    if strategy == "forward":
        assert (ends[fitting] <= left).all()
    return np.flatnonzero(fitting), np.flatnonzero(validation)


partitions, coverage = [], []
for year in range(2015, 2023):
    for strategy in ("forward", "loyo"):
        fitting, validation = year_partition(year, strategy)
        fit = train.iloc[fitting]
        partitions.append({"year": year, "strategy": strategy, "fitting_cases": len(fitting),
            "scoring_cases": len(validation), "future_fitting_cases": int((fit.starttime.dt.year > year).sum()),
            "training_years": ",".join(map(str, sorted(fit.starttime.dt.year.unique())))})
        if len(fitting):
            for column in COLUMNS:
                lo, hi = np.quantile(features.iloc[fitting][column], [.01, .99])
                values = features.iloc[validation][column]
                coverage.append({"year": year, "strategy": strategy, "feature": column,
                                 "fit_q01": lo, "fit_q99": hi,
                                 "scoring_fraction_outside_fit_1_99": float(((values < lo) | (values > hi)).mean())})
        pd.DataFrame({"Run No": train.iloc[fitting]["Run No"], "starttime": train.iloc[fitting].starttime}).to_csv(
            OUT / f"{strategy}_{year}_fitting_cases.csv", index=False)
        pd.DataFrame({"Run No": train.iloc[validation]["Run No"], "starttime": train.iloc[validation].starttime}).to_csv(
            OUT / f"{strategy}_{year}_scoring_cases.csv", index=False)
assert np.array_equal(year_partition(2022, "forward")[0], year_partition(2022, "loyo")[0])
assert [(len(year_partition(y, "forward")[0]), len(year_partition(y, "forward")[1])) for y in [2020, 2021]] == [(1820, 360), (2186, 359)]
partition_summary = pd.DataFrame(partitions)
coverage = pd.DataFrame(coverage)
partition_summary.to_csv(OUT / "partition_summary.csv", index=False)
coverage.to_csv(OUT / "input_percentile_coverage.csv", index=False)
display(partition_summary.loc[partition_summary.year >= 2019])
display(coverage.loc[(coverage.year >= 2020) & coverage.feature.isin(
    ["weekly_mean_price_EUR_MWh", "mean_terminal_local_value_EUR_MWh"])].round(4))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), layout="constrained")
cmap = ListedColormap(["#f0f2f5", "#537592", "#37a4a0", "#e5a331", "#b4bac2"])
for ax, strategy in zip(axes, ("forward", "loyo")):
    matrix = np.zeros((4, len(train)), dtype=int)
    for row, year in enumerate(range(2019, 2023)):
        fitting, validation = year_partition(year, strategy)
        matrix[row, fitting] = np.where(train.iloc[fitting].starttime.dt.year > year, 2, 1)
        matrix[row, validation] = 3
        # Purged/trimmed boundary cases, including within-year horizon-end trimming.
        start = pd.Timestamp(f"{year}-01-01", tz="UTC")
        end = pd.Timestamp(f"{year+1}-01-01", tz="UTC")
        around = (train.starttime >= start-pd.Timedelta(days=6)) & (train.starttime < end)
        matrix[row, np.flatnonzero(around & (matrix[row] == 0))] = 4
    ax.imshow(matrix, aspect="auto", cmap=cmap, norm=BoundaryNorm(np.arange(-.5, 5), 5), interpolation="nearest")
    ticks = [int(np.flatnonzero(train.starttime.dt.year == y)[0]) for y in range(2015, 2023)]
    ax.set(xticks=ticks, xticklabels=range(2015, 2023), yticks=range(4), yticklabels=range(2019, 2023),
           title="Forward: earlier years only" if strategy == "forward" else "LOYO: earlier and later years", ylabel="Scoring year")
fig.suptitle("Same scoring cases; different training information", fontsize=16, fontweight="bold")
fig.legend(handles=[Patch(color=cmap(i), label=label) for i, label in enumerate(
    ["Unused", "Earlier training", "Later training", "Scoring", "Boundary excluded"])],
    loc="outside lower center", ncol=5, frameon=False)
fig.savefig(OUT / "forward_vs_loyo.png", dpi=160)
plt.show()

manifest = {"created_utc": datetime.now(timezone.utc).isoformat(), "fitting_performed": False,
    "weights_loaded": False, "new_loyo_scores": False, "model_notebook_modified": False,
    "case_summary_rows": len(features), "label_years": list(range(2015,2023)),
    "input_sha256": {k: hashlib.sha256(v.read_bytes()).hexdigest() for k,v in FILES.items()},
    "versions": {"python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__}}
(OUT / "manifest.json").write_text(json.dumps(manifest, indent=2)+"\n", encoding="utf-8")
print("Audit passed; no models fitted. Results:", OUT)


## Interpretation and next experiment

A distinct 2022 price regime is a covariate observation, not proof that only 2022
is difficult. Existing v32 chronological scores must be read with their own case
counts and training histories. No new LOYO AUC has been measured here.

Use purged forward years as the main estimate of post-2022 transfer. LOYO can be
a supplementary regime-transfer stress test, reporting every year separately,
especially 2022. Under these partitions 2022 LOYO and forward use **identical**
fitting cases: no later labeled year exists. Improvements in earlier LOYO folds
may reflect extra data and later-regime exposure. Never drop 2022 merely to raise
the mean, or choose a validation method because its score is close to public AUC.

Before any trained comparison, freeze the same recipe and define separate inner
stopping/tuning and outer scoring data. Do not use held-year labels for epoch
selection, scaling or calibrating cross-year predictions. Report annual micro AUC,
worst year and per-unit diagnostics; a pooled out-of-fold micro AUC additionally
compares scores from different fitted models and can change with their scales.

Method reference: [scikit-learn TimeSeriesSplit](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.TimeSeriesSplit.html).
The final TCN recipe, submission and full-data training remain unchanged.